# Практика · Мовна модель: наступне слово

> ⏱ Зошит навчає **три** нейромережі (три зерна одної й тієї самої моделі) і будує
> пʼять n-грамових. Заміряно: **близько пʼяти хвилин процесорного часу** на чотирьох
> ядрах без відеокарти, з потоками, зафіксованими в один. Більшість цього часу —
> три навчання приблизно по 95 секунд.

Лекція: [lecture.html](lecture.html) · Домашнє: [homework.html](homework.html) ·
Тест: [quiz.html](quiz.html)

Що зробимо:

1. Зберемо корпус і поділимо його на навчальну й перевірну частини.
2. Напишемо перплексію своїми руками й перевіримо її проти `torch`.
3. Піднімемось сходинками: рівномірна → уніграма → біграма → триграма.
4. Заміряємо, скільки перевірних біграм модель не бачила жодного разу.
5. Підберемо добавку **чесно** — на добірній частині, вирізаній із навчальної.
6. Навчимо нейромережу з вікном у два слова, три зерна.
7. Складемо все в одну таблицю й порахуємо, що зсунуло перплексію сильніше.

Кожне число лекції надруковане тут.

## 0 · Фіксуємо потоки до імпорту numpy

Без цього процесорний час бреше — і сильніше, ніж стінний. Потоки OpenMP крутяться
в очікуванні, і це очікування рахується як робота. Тому змінні середовища ставимо
**до** `import numpy`, а не після: після імпорту вони вже нічого не змінять.

In [ ]:
import os
# потоки фіксуємо ДО імпорту numpy — інакше замір часу буде неправильний у рази
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import time, math, glob, gettext, re
import numpy as np
from collections import Counter, defaultdict
import torch
import torch.nn as nn

torch.set_num_threads(1)          # те саме для torch
START = time.process_time()       # міряємо процесорний час, а не стінний

print('numpy', np.__version__, '· torch', torch.__version__)
print('потоків torch:', torch.get_num_threads())

## 1 · Корпус

Той самий, що й у попередніх темах: українські переклади інтерфейсів програм,
встановлених у системі. Файли `.mo` лежать у `/usr/share/locale/uk/LC_MESSAGES/`.

Токенізатор — канонічний для курсу. Апостроф у ньому не літера слова, а **звʼязка**
між двома пробігами літер, тому `зʼєднання` лишається одним токеном.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
tokenizer = re.compile(TOKEN_PATTERN)


def load_corpus():
    "Читає українські переклади з усіх .mo-каталогів системи."
    texts = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
            for source, target in catalog._catalog.items():
                # беремо лише справжні переклади: рядок, довший за 30 символів,
                # і не службовий заголовок каталогу
                if isinstance(source, str) and isinstance(target, str) \
                   and len(target) > 30 and 'Project-Id' not in target:
                    texts.append(target)
        except Exception:
            pass          # пошкоджений або нечитабельний каталог просто пропускаємо
    return texts


documents = load_corpus()
if len(documents) < 1000:
    raise RuntimeError('українських перекладів у системі надто мало — '
                       'встанови мовний пакет uk або запусти зошит на іншій машині')

sentences = [tokenizer.findall(text.lower()) for text in documents]
lengths = np.array([len(s) for s in sentences])

print('документів:', len(documents))
print('слововживань:', int(lengths.sum()))
print('словоформ:', len({w for s in sentences for w in s}))
print('довжина речення: медіана %d, 90-й перцентиль %d, найдовше %d'
      % (np.median(lengths), np.percentile(lengths, 90), lengths.max()))

## 2 · Поділ на навчальну й перевірну частини

Речення коротші за два слова нічого не дають мовній моделі, а довші за тридцять —
рідкісні викиди. Лишаємо діапазон `2 ≤ довжина ≤ 30` і ділимо 90 на 10 із зерном 0.

Поділ **обовʼязково випадковий**: у натуральному порядку файлів речення згруповані
за програмами, і перевірна частина складалася б із кількох останніх програм.

In [ ]:
kept = [s for s in sentences if 2 <= len(s) <= 30]

rng = np.random.default_rng(0)
order = rng.permutation(len(kept))
n_val = len(kept) // 10
val_positions = set(order[:n_val].tolist())

train_sents = [kept[i] for i in range(len(kept)) if i not in val_positions]
val_sents = [kept[i] for i in range(len(kept)) if i in val_positions]

print('речень після відсіву 2..30:', len(kept))
print('навчальних речень:', len(train_sents))
print('перевірних речень:', len(val_sents))

## 3 · Словник

Словник будуємо **тільки за навчальною частиною** — інакше модель підглядала б у
перевірну. Слово, що трапилось менше пʼяти разів, замінюємо на `<unk>`.

Три службові позиції: `<pad>` (знадобиться наступним темам блоку), `<eos>` — кінець
речення, і `<unk>` — невідоме слово. `<eos>` дописуємо в кінець кожного речення, щоб
модель уміла сказати «текст закінчився» і щоб речення різної довжини порівнювались
чесно.

In [ ]:
MIN_COUNT = 5

train_counts = Counter()
for s in train_sents:
    train_counts.update(s)

itos = ['<pad>', '<eos>', '<unk>'] + sorted(w for w, c in train_counts.items()
                                            if c >= MIN_COUNT)
stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)
EOS, UNK = 1, 2


def encode(sents):
    "Слово → номер; невідоме → <unk>; у кінець кожного речення дописуємо <eos>."
    return [[stoi.get(w, UNK) for w in s] + [EOS] for s in sents]


train_ids = encode(train_sents)
val_ids = encode(val_sents)

n_train_tokens = sum(len(x) for x in train_ids)
n_val_tokens = sum(len(x) for x in val_ids)
n_val_words = sum(len(s) for s in val_sents)
n_val_unk = sum(1 for x in val_ids for i in x if i == UNK)

print('словник разом із <pad>, <eos>, <unk>:', VOCAB)
print('навчальних токенів разом з <eos>:', n_train_tokens)
print('перевірних токенів разом з <eos>:', n_val_tokens)
print('перевірних слововживань без <eos>:', n_val_words)
print('серед них <unk>: %d = %.2f %%' % (n_val_unk, 100 * n_val_unk / n_val_words))

## 4 · Перплексія своїми руками

Перплексія — це `exp` від середньої відʼємної логарифмічної ймовірності:

```
перплексія = exp( − (1/N) · Σ log P(слово | контекст) )
```

`N` — скільки разів модель робила передбачення. Не «скільки слів у тексті», а саме
**скільки передбачень**, тобто разом із `<eos>`.

Спершу перевіримо формулу на трьох числах, які легко порахувати в голові:
ймовірності 0.5, 0.25 і 0.125 — це 1/2, 1/4 і 1/8, і середнє геометричне чисел
2, 4, 8 дорівнює 4.

In [ ]:
def perplexity_from_probs(probs):
    "Перплексія за списком ймовірностей, які модель дала правильним словам."
    return math.exp(-sum(math.log(p) for p in probs) / len(probs))


example_probs = [0.5, 0.25, 0.125]

# показуємо всі проміжні числа, щоб формулу можна було перевірити руками
for p in example_probs:
    print('log %-6s = %.4f' % (p, math.log(p)))
mean_loss = -sum(math.log(p) for p in example_probs) / len(example_probs)
print('сума логарифмів зі знаком мінус: %.4f' % (-sum(math.log(p) for p in example_probs)))
print('середня логарифмічна втрата:     %.4f' % mean_loss)

hand = perplexity_from_probs(example_probs)
print('перплексія від 0.5, 0.25, 0.125: %.4f' % hand)
print('середнє геометричне 2, 4, 8:     %.4f' % ((2 * 4 * 8) ** (1 / 3)))
assert abs(hand - 4.0) < 1e-9, 'формула перплексії розійшлася з арифметикою!'
print('✅ формула збігається з ручним рахунком')

## 5 · Нульова сходинка: рівномірна модель

Модель, яка нічого не знає, дає всім словам `1/VOCAB`. Її перплексія дорівнює
розміру словника — це можна вивести, але ми ще й порахуємо.

In [ ]:
uniform_prob = 1.0 / VOCAB
uniform_ppl = perplexity_from_probs([uniform_prob] * 1000)

print('ймовірність кожного слова: %.6f' % uniform_prob)
print('перплексія рівномірної моделі: %.2f' % uniform_ppl)
print('розмір словника:              %d' % VOCAB)
assert abs(uniform_ppl - VOCAB) < 1e-6
print('✅ перплексія рівномірної моделі дорівнює розміру словника')

## 6 · Уніграма: слова нерівні

Ймовірність слова — його частка в навчальному тексті. Контекст не дивимо взагалі.

Одиниця в чисельнику й `VOCAB` у знаменнику — добавка: без неї слово з нульовим
лічильником дістало б рівно нуль, а логарифм нуля не існує.

In [ ]:
unigram_counts = np.zeros(VOCAB, dtype=np.int64)
for x in train_ids:
    for i in x:
        unigram_counts[i] += 1

unigram_probs = (unigram_counts + 1.0) / (unigram_counts.sum() + VOCAB)

top = np.argsort(-unigram_counts)[:9]
print('найчастіші слова навчальної частини:')
for i in top:
    print('   %-14s %7d   %.5f' % (itos[i], unigram_counts[i], unigram_probs[i]))

log_probs = np.log(unigram_probs)
unigram_loss = -sum(log_probs[i] for x in val_ids for i in x) / n_val_tokens
unigram_ppl = math.exp(unigram_loss)
print()
print('перплексія уніграми: %.2f' % unigram_ppl)

## 7 · Біграма: попереднє слово

Рахуємо, скільки разів слово `b` йшло одразу після слова `a`. Контекстом для першого
слова речення служить `<eos>` попереднього — тобто «початок речення».

In [ ]:
def count_bigrams(seqs):
    "Повертає лічильники пар і лічильники контекстів."
    pair_counts = defaultdict(Counter)
    context_totals = np.zeros(VOCAB, dtype=np.int64)
    for x in seqs:
        previous = EOS                  # початок речення = після <eos>
        for i in x:
            pair_counts[previous][i] += 1
            context_totals[previous] += 1
            previous = i
    return pair_counts, context_totals


bigram_counts, context_totals = count_bigrams(train_ids)
print('різних пар слів у навчальній частині:', sum(len(c) for c in bigram_counts.values()))
print('усіх мислимих пар:', VOCAB * VOCAB)
print('бачили: %.4f %%' % (100 * sum(len(c) for c in bigram_counts.values()) / VOCAB ** 2))

print()
for word in ['<eos>', 'не', 'файл', 'для', 'помилка']:
    p = stoi[word]
    best = bigram_counts[p].most_common(9)
    shown = ', '.join('%s %d' % (itos[i], c) for i, c in best)
    print('після «%s» (%d разів): %s' % (word, context_totals[p], shown))

# ті самі лічильники як ймовірності — саме ці числа стоять у лекції
print()
after_ne = stoi['не']
for i, c in bigram_counts[after_ne].most_common(3):
    print('P(%s | не) = %d / %d = %.4f'
          % (itos[i], c, context_totals[after_ne], c / context_totals[after_ne]))
after_error = stoi['помилка']
i, c = bigram_counts[after_error].most_common(1)[0]
print('P(%s | помилка) = %d / %d = %.4f'
      % (itos[i], c, context_totals[after_error], c / context_totals[after_error]))
print('для порівняння, уніграма: P(вдалося) = %.5f' % unigram_probs[stoi['вдалося']])
print('контекст підняв ймовірність правильного слова у %.1f раза'
      % ((bigram_counts[after_ne][stoi['вдалося']] / context_totals[after_ne])
         / unigram_probs[stoi['вдалося']]))

## 8 · Скільки пар модель не бачила

Це головне число теми. Рахуємо його **двома** способами, бо вони дають різні
відповіді, і різниця повчальна.

**Спосіб перший** — так, як бачить модель: рідкісні слова вже зведені до `<unk>`,
контекстом першого слова служить `<eos>`.

**Спосіб другий** — пари справжніх слів усередині речення, без поблажки `<unk>`.

In [ ]:
seen_pairs = set()
for x in train_ids:
    previous = EOS
    for i in x:
        seen_pairs.add((previous, i))
        previous = i

val_pairs = []
for x in val_ids:
    previous = EOS
    for i in x:
        val_pairs.append((previous, i))
        previous = i

missing_model = sum(1 for pair in val_pairs if pair not in seen_pairs)
print('перевірних біграм усього: %d' % len(val_pairs))
print('з них не траплялись:      %d = %.2f %%'
      % (missing_model, 100 * missing_model / len(val_pairs)))

seen_word_pairs = set()
for s in train_sents:
    for a, b in zip(s, s[1:]):
        seen_word_pairs.add((a, b))

word_pairs = [(a, b) for s in val_sents for a, b in zip(s, s[1:])]
missing_words = sum(1 for pair in word_pairs if pair not in seen_word_pairs)
print()
print('пар справжніх слів усього: %d' % len(word_pairs))
print('з них не траплялись:       %d = %.2f %%'
      % (missing_words, 100 * missing_words / len(word_pairs)))

## 9 · Добавка, і чому її не можна підбирати на перевірній частині

Добавка `k` рятує від нулів:

```
P(b | a) = ( c(a,b) + k ) / ( c(a) + k · VOCAB )
```

Але `k` треба звідкись узяти. Якщо перебирати його прямо на перевірній частині, ми
виміряємо не якість моделі, а власну вправність у підгонці. Тому вирізаємо з
**навчальної** частини ще одну — **добірну** (dev), і перебираємо на ній.

Лічильники для добору рахуємо без добірної частини, інакше вона теж стане «баченою».

In [ ]:
dev_rng = np.random.default_rng(1)
dev_order = dev_rng.permutation(len(train_ids))
n_dev = len(train_ids) // 10
dev_positions = set(dev_order[:n_dev].tolist())

fit_ids = [train_ids[i] for i in range(len(train_ids)) if i not in dev_positions]
dev_ids = [train_ids[i] for i in range(len(train_ids)) if i in dev_positions]

fit_bigrams, fit_contexts = count_bigrams(fit_ids)

dev_pairs = []
for x in dev_ids:
    previous = EOS
    for i in x:
        dev_pairs.append((previous, i))
        previous = i

print('підгінних речень: %d · добірних: %d' % (len(fit_ids), len(dev_ids)))
print('добірних біграм: %d' % len(dev_pairs))

In [ ]:
def bigram_perplexity(pairs, pair_counts, context_totals, k):
    "Перплексія біграми з добавкою k на заданому списку пар."
    total = 0.0
    for previous, word in pairs:
        row = pair_counts.get(previous)
        c = row.get(word, 0) if row is not None else 0
        total += math.log((c + k) / (context_totals[previous] + k * VOCAB))
    return math.exp(-total / len(pairs))


GRID = [3.0, 2.0, 1.0, 0.5, 0.3, 0.2, 0.1, 0.05, 0.03, 0.02, 0.01, 0.005,
        0.003, 0.002, 0.001]

print('   k        добірна   перевірна')
dev_scores = []
val_scores = []
for k in GRID:
    on_dev = bigram_perplexity(dev_pairs, fit_bigrams, fit_contexts, k)
    on_val = bigram_perplexity(val_pairs, bigram_counts, context_totals, k)
    dev_scores.append(on_dev)
    val_scores.append(on_val)
    print('%7.4g  %9.2f  %10.2f' % (k, on_dev, on_val))

best_index = int(np.argmin(dev_scores))
BEST_K = GRID[best_index]
bigram_ppl = val_scores[best_index]
print()
print('добірна частина обрала k = %g' % BEST_K)
print('перплексія на перевірній: %.2f' % bigram_ppl)
print('та сама біграма з добавкою 1.0: %.2f' % val_scores[GRID.index(1.0)])
print('розмах від однієї ручки: %.2f пункта'
      % (val_scores[GRID.index(1.0)] - bigram_ppl))

## 10 · Куди йде втрата

Розбираємо перевірний текст на дві купи — знайомі пари й незнайомі — і рахуємо
перплексію кожної окремо. Плюс дивимось, яку частку ймовірнісної маси добавка
роздає словам, яких після цього контексту не бачили жодного разу.

In [ ]:
print('   k    знайомі  незнайомі  частка втрати  маса небаченим')
for k in [1.0, 0.5, 0.1, 0.05, 0.01, 0.003]:
    n_seen = n_unseen = 0
    loss_seen = loss_unseen = 0.0
    for previous, word in val_pairs:
        row = bigram_counts.get(previous)
        c = row.get(word, 0) if row is not None else 0
        log_p = math.log((c + k) / (context_totals[previous] + k * VOCAB))
        if c > 0:
            n_seen += 1
            loss_seen -= log_p
        else:
            n_unseen += 1
            loss_unseen -= log_p

    # скільки маси йде небаченим продовженням у середньому по контекстах
    mass_total = 0.0
    n_contexts = 0
    for previous in range(VOCAB):
        if context_totals[previous] == 0:
            continue
        unseen_words = VOCAB - len(bigram_counts[previous])
        mass_total += unseen_words * k / (context_totals[previous] + k * VOCAB)
        n_contexts += 1

    print('%6.4g  %8.2f  %9.2f  %12.2f %%  %13.2f %%'
          % (k, math.exp(loss_seen / n_seen), math.exp(loss_unseen / n_unseen),
             100 * loss_unseen / (loss_seen + loss_unseen),
             100 * mass_total / n_contexts))

print()
print('знайомих пар: %d · незнайомих: %d' % (n_seen, n_unseen))

## 11 · Інтерполяція: позичити в уніграми

Добавка роздає незнайомим парам порівну. Але ми вже знаємо, що слова нерівні —
тож замість рівної роздачі змішаємо біграму з уніграмою:

```
P(b | a) = λ · c(a,b)/c(a) + (1 − λ) · P_уні(b)
```

λ добираємо так само чесно — на добірній частині.

In [ ]:
fit_unigram = np.zeros(VOCAB, dtype=np.int64)
for x in fit_ids:
    for i in x:
        fit_unigram[i] += 1
fit_unigram_probs = (fit_unigram + 1.0) / (fit_unigram.sum() + VOCAB)


def interpolated_perplexity(pairs, pair_counts, context_totals, backoff, lam):
    "Суміш біграми з уніграмою: λ довіри біграмі, решта — уніграмі."
    total = 0.0
    for previous, word in pairs:
        row = pair_counts.get(previous)
        c = row.get(word, 0) if row is not None else 0
        denominator = context_totals[previous]
        p_bigram = c / denominator if denominator else 0.0
        total += math.log(lam * p_bigram + (1 - lam) * backoff[word])
    return math.exp(-total / len(pairs))


LAMBDAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]

print('  λ        добірна   перевірна')
dev_lam = []
val_lam = []
for lam in LAMBDAS:
    on_dev = interpolated_perplexity(dev_pairs, fit_bigrams, fit_contexts,
                                     fit_unigram_probs, lam)
    on_val = interpolated_perplexity(val_pairs, bigram_counts, context_totals,
                                     unigram_probs, lam)
    dev_lam.append(on_dev)
    val_lam.append(on_val)
    print('%5.2f  %9.2f  %10.2f' % (lam, on_dev, on_val))

best_lam_index = int(np.argmin(dev_lam))
BEST_LAMBDA = LAMBDAS[best_lam_index]
interpolated_ppl = val_lam[best_lam_index]
print()
print('добірна частина обрала λ = %g → перплексія %.2f' % (BEST_LAMBDA, interpolated_ppl))
print('це на %.2f пункта краще за найкращу добавку' % (bigram_ppl - interpolated_ppl))

## 12 · Триграма: більше контексту — гірший результат

Тепер дивимось на два попередні слова. Логіка та сама, добір той самий.

In [ ]:
def count_trigrams(seqs):
    pair_counts = defaultdict(Counter)
    context_totals = Counter()
    for x in seqs:
        a, b = EOS, EOS
        for i in x:
            pair_counts[(a, b)][i] += 1
            context_totals[(a, b)] += 1
            a, b = b, i
    return pair_counts, context_totals


def trigram_list(seqs):
    out = []
    for x in seqs:
        a, b = EOS, EOS
        for i in x:
            out.append((a, b, i))
            a, b = b, i
    return out


trigram_counts, trigram_contexts = count_trigrams(train_ids)
fit_trigrams, fit_tri_contexts = count_trigrams(fit_ids)
val_triples = trigram_list(val_ids)
dev_triples = trigram_list(dev_ids)

seen_triples = sum(1 for a, b, w in val_triples
                   if trigram_counts.get((a, b), {}).get(w, 0) > 0)
seen_contexts = sum(1 for a, b, w in val_triples if (a, b) in trigram_contexts)

print('різних триграм у навчальній частині:',
      sum(len(c) for c in trigram_counts.values()))
print('перевірних триграм, яких не бачили: %.2f %%'
      % (100 * (1 - seen_triples / len(val_triples))))
print('перевірних двослівних контекстів, яких не бачили: %.2f %%'
      % (100 * (1 - seen_contexts / len(val_triples))))

In [ ]:
def trigram_perplexity(triples, counts, contexts, k):
    total = 0.0
    for a, b, word in triples:
        row = counts.get((a, b))
        c = row.get(word, 0) if row is not None else 0
        total += math.log((c + k) / (contexts.get((a, b), 0) + k * VOCAB))
    return math.exp(-total / len(triples))


TRI_GRID = [0.1, 0.03, 0.01, 0.003, 0.001, 0.0005, 0.0002]

print('     k        добірна   перевірна')
tri_dev = []
tri_val = []
for k in TRI_GRID:
    on_dev = trigram_perplexity(dev_triples, fit_trigrams, fit_tri_contexts, k)
    on_val = trigram_perplexity(val_triples, trigram_counts, trigram_contexts, k)
    tri_dev.append(on_dev)
    tri_val.append(on_val)
    print('%8.4g  %9.2f  %10.2f' % (k, on_dev, on_val))

best_tri = int(np.argmin(tri_dev))
trigram_ppl = tri_val[best_tri]
print()
print('добірна частина обрала k = %g → перплексія %.2f' % (TRI_GRID[best_tri], trigram_ppl))
print('біграма з підібраною добавкою давала %.2f' % bigram_ppl)
print('удвічі довший контекст коштував %.2f пункта ПОГІРШЕННЯ'
      % (trigram_ppl - bigram_ppl))

## 13 · Нейромережа з вікном у два слова

Контекст у неї такий самий, як у триграми: два попередні слова. Різниця — у тому,
як вона його використовує. Триграма шукає точну комбінацію в таблиці; мережа
замінює кожне слово ембедингом, склеює два ембединги в один вектор і стискає його
через прихований шар. Схожі слова дають схожі вектори, тож незнайоме поєднання
знайомих слів перестає бути порожнім місцем.

Одне навчання — приблизно **95 секунд** процесорного часу. Зерен три, тож ця
клітинка йде близько пʼяти хвилин.

In [ ]:
CONTEXT = 2          # скільки попередніх слів бачить мережа
EMBEDDING = 64       # довжина вектора одного слова
HIDDEN = 64          # ширина прихованого шару
BATCH = 1024
LEARNING_RATE = 8e-3


def make_windows(seqs):
    "Кожен токен стає прикладом: два попередні слова → цей токен."
    inputs, targets = [], []
    for x in seqs:
        history = [EOS] * CONTEXT
        for i in x:
            inputs.append(list(history))
            targets.append(i)
            history = history[1:] + [i]
    return torch.tensor(inputs), torch.tensor(targets)


train_x, train_y = make_windows(train_ids)
val_x, val_y = make_windows(val_ids)
print('навчальних вікон:', len(train_x), '· перевірних:', len(val_x))


class WindowLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB, EMBEDDING)
        self.hidden = nn.Linear(CONTEXT * EMBEDDING, HIDDEN)
        self.output = nn.Linear(HIDDEN, VOCAB)

    def forward(self, context_words):
        # два ембединги склеюємо в один вектор із 128 чисел
        flat = self.embedding(context_words).flatten(1)
        return self.output(torch.tanh(self.hidden(flat)))


print('параметрів у моделі:',
      sum(p.numel() for p in WindowLanguageModel().parameters()))

In [ ]:
def evaluate(model, xs, ys):
    "Перплексія моделі на заданих вікнах."
    model.eval()
    total = 0.0
    with torch.no_grad():
        for start in range(0, len(xs), 8192):
            logits = model(xs[start:start + 8192])
            total += nn.functional.cross_entropy(
                logits, ys[start:start + 8192], reduction='sum').item()
    return math.exp(total / len(ys))


# для проміжних замірів беремо підвибірку — щоб не платити за повний прогін чотири рази
sample = torch.randperm(len(val_x),
                        generator=torch.Generator().manual_seed(7))[:20000]
sample_x, sample_y = val_x[sample], val_y[sample]

CHECKPOINTS = [0.25, 0.5, 0.75, 1.0]
curves, finals, durations = [], [], []

for seed in (0, 1, 2):
    torch.manual_seed(seed)
    model = WindowLanguageModel()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    shuffle = torch.randperm(len(train_x),
                             generator=torch.Generator().manual_seed(seed))
    n_batches = (len(shuffle) + BATCH - 1) // BATCH
    marks = [int(n_batches * c) for c in CHECKPOINTS]

    started = time.process_time()
    curve = []
    model.train()
    for step in range(n_batches):
        batch = shuffle[step * BATCH:(step + 1) * BATCH]
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(train_x[batch]), train_y[batch])
        loss.backward()
        optimizer.step()
        if step + 1 in marks:
            last = (step + 1 == marks[-1])
            curve.append(evaluate(model, val_x, val_y) if last
                         else evaluate(model, sample_x, sample_y))
            model.train()
    spent = time.process_time() - started

    curves.append(curve)
    finals.append(curve[-1])
    durations.append(spent)
    print('зерно %d: перплексія %.2f · крива %s · %.1f с процесорних'
          % (seed, curve[-1], ' '.join('%.2f' % c for c in curve), spent))

finals = np.array(finals)
network_ppl = float(finals.mean())
print()
print('мережа: %.2f ±%.2f  (від %.2f до %.2f)'
      % (network_ppl, finals.std(ddof=1), finals.min(), finals.max()))
print('навчання: %.1f ±%.1f с процесорних на зерно'
      % (np.mean(durations), np.std(durations, ddof=1)))

## 14 · Перевірка: наша перплексія = те, що дає torch

Обовʼязковий крок. Беремо одне речення з перевірної частини, рахуємо ймовірність
кожного слова власними руками через `softmax`, зводимо їх у перплексію нашою
функцією — і звіряємо з `cross_entropy` від `torch`.

In [ ]:
# беремо перше речення з 6-7 слів, у якому всі слова відомі моделі
example_sent, example_ids = None, None
for s, x in zip(val_sents, val_ids):
    if 6 <= len(s) <= 7 and all(stoi.get(w, UNK) != UNK for w in s):
        example_sent, example_ids = s, x
        break

print('приклад:', ' '.join(example_sent))
print()

model.eval()
probabilities = {'рівномірна': [], 'уніграма': [], 'біграма': [], 'мережа': []}
history = [EOS] * CONTEXT
previous = EOS
with torch.no_grad():
    for i in example_ids:
        probabilities['рівномірна'].append(1.0 / VOCAB)
        probabilities['уніграма'].append(float(unigram_probs[i]))
        c = bigram_counts[previous].get(i, 0)
        probabilities['біграма'].append(
            (c + BEST_K) / (context_totals[previous] + BEST_K * VOCAB))
        logits = model(torch.tensor([history]))
        probabilities['мережа'].append(float(torch.softmax(logits, 1)[0, i]))
        history = history[1:] + [i]
        previous = i

labels = list(example_sent) + ['<eos>']
print('%-16s %10s %10s %10s %10s' % ('слово', 'рівном.', 'уніграма', 'біграма', 'мережа'))
for j, word in enumerate(labels):
    print('%-16s %10.5f %10.5f %10.5f %10.5f'
          % (word, probabilities['рівномірна'][j], probabilities['уніграма'][j],
             probabilities['біграма'][j], probabilities['мережа'][j]))

In [ ]:
our_ppl = perplexity_from_probs(probabilities['мережа'])

# те саме, але через готову cross_entropy: збираємо контексти цього ж речення
contexts = [[EOS, EOS]]
for j in range(1, len(example_ids)):
    contexts.append([EOS if j == 1 else example_ids[j - 2], example_ids[j - 1]])
with torch.no_grad():
    reference = math.exp(nn.functional.cross_entropy(
        model(torch.tensor(contexts)), torch.tensor(example_ids)).item())

print('наша перплексія речення:  %.4f' % our_ppl)
print('через torch.cross_entropy: %.4f' % reference)
assert abs(our_ppl - reference) / reference < 1e-4, 'розрахунок розійшовся!'
print('✅ збігається')

## 15 · Усі сходинки в одній таблиці

І головне питання теми: що зсунуло перплексію сильніше — вибір методу чи
налаштування того, що вже було?

In [ ]:
bigram_default = val_scores[GRID.index(1.0)]
bigram_tenth = val_scores[GRID.index(0.1)]

table = [
    ('рівномірна',                        '—',       uniform_ppl),
    ('біграма, добавка 1.0',              '1 слово', bigram_default),
    ('уніграма',                          '—',       unigram_ppl),
    ('триграма, добавка %g' % TRI_GRID[best_tri], '2 слова', trigram_ppl),
    ('біграма, добавка 0.1',              '1 слово', bigram_tenth),
    ('мережа, вікно 2 слова',             '2 слова', network_ppl),
    ('біграма, добавка %g' % BEST_K,      '1 слово', bigram_ppl),
    ('біграма з уніграмою, λ = %g' % BEST_LAMBDA, '1 слово', interpolated_ppl),
]

print('%-34s %-9s %10s' % ('модель', 'контекст', 'перплексія'))
for name, context, value in table:
    print('%-34s %-9s %10.2f' % (name, context, value))

tuning_gain = bigram_default - bigram_ppl
method_gain = bigram_default - network_ppl
print()
print('одна ручка (добавка 1.0 → %g):        %8.2f пункта' % (BEST_K, tuning_gain))
print('зміна методу (біграма 1.0 → мережа):  %8.2f пункта' % method_gain)
print('від найкращої біграми до мережі:      %8.2f пункта'
      % (bigram_ppl - network_ppl))
print('від найкращої біграми до інтерполяції:%8.2f пункта'
      % (bigram_ppl - interpolated_ppl))
print()
print('мережа проти триграми з тим самим контекстом: %.2f проти %.2f, тобто у %.2f раза краще'
      % (network_ppl, trigram_ppl, trigram_ppl / network_ppl))

In [ ]:
print('увесь зошит: %.1f с процесорного часу' % (time.process_time() - START))

## Завдання

**🟢 Рівень 1.** Обмеж `glob` сорока першими файлами й повтори сходинки: рівномірна,
уніграма, біграма з добавками 1.0, 0.1, 0.01, 0.003. Чи лишилась біграма з добавкою
1.0 гіршою за уніграму на меншому корпусі?

**🟡 Рівень 2.** Підбери `k` двічі: чесно (на добірній частині) і нечесно (прямо на
перевірній). Назви різницю в пунктах. Потім поміняй розмір добірної частини —
1 %, 5 %, 10 %, 30 % — і подивись, з якого розміру обране `k` перестає стрибати.

**🔴 Рівень 3.** Реалізуй трирівневу інтерполяцію `λ₃·P₃ + λ₂·P₂ + λ₁·P₁`, підбери
трійку на добірній частині й порівняй із трьома числами цього зошита. Окремо
надрукуй, яку частку перевірних токенів обслужив кожен рівень.

Повний текст завдань — у [homework.html](homework.html).